In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [17]:

!pip install datasets==2.19.0
!pip install evaluate==0.4.2
!pip install jiwer==3.0.4
!pip install librosa==0.10.1
!pip install soundfile
!pip install torchaudio
!pip install google-genai
!pip install tqdm
print('✅ All packages installed successfully!')

✅ All packages installed successfully!


In [18]:
import torch
import numpy as np
import pandas as pd
import warnings
import gc
import os
from google import genai
import tempfile
import soundfile as sf
from tqdm import tqdm

warnings.filterwarnings('ignore')

from datasets import load_dataset, Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# ── GPU Check ──────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"🖥️  Device  : {device.upper()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"🎮 GPU     : {gpu.name}")
    print(f"💾 VRAM    : {gpu.total_memory / 1e9:.1f} GB")
print(f"🔢 PyTorch : {torch.__version__}")

🖥️  Device  : CUDA
🎮 GPU     : Tesla T4
💾 VRAM    : 15.6 GB
🔢 PyTorch : 2.10.0+cu128


In [19]:
print("📥 Loading ujs/hinglish-compressed dataset...")
dataset = load_dataset("ujs/hinglish-compressed", trust_remote_code=True)
print(f"✅ Dataset loaded: {dataset}")

# Resample all audio to 16kHz (required by Whisper)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
print("✅ Audio resampled to 16kHz")

# ── Fix the SAME 10 test samples for ALL evaluations ──────────────────────────
# Using indices 0–9 from test split for reproducibility
TEST_INDICES = list(range(10))
test_samples = [dataset["test"][i] for i in TEST_INDICES]

print("\n📋 The 10 Fixed Test Samples (Reference Transcriptions):")
print("-" * 70)
for i, s in enumerate(test_samples):
    print(f"[{i:02d}] {s['sentence'][:80]}")

# Store references
REFERENCES = [s['sentence'] for s in test_samples]

📥 Loading ujs/hinglish-compressed dataset...
✅ Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'sentence'],
        num_rows: 52825
    })
    test: Dataset({
        features: ['path', 'audio', 'sentence'],
        num_rows: 3136
    })
})
✅ Audio resampled to 16kHz

📋 The 10 Fixed Test Samples (Reference Transcriptions):
----------------------------------------------------------------------
[00] लिबर ऑफिस impress में एक प्रस्तुति document बनाना और बुनियादी formatting के इस s
[01] इस tutorial में हम impress window के भागों के बारे में सीखेंगे और कैसे स्लाइड इन
[02] यहाँ हम अपने ऑपरेटिंग सिस्टम के रूप में gnu/linux और लिबरऑफिस वर्जन 334 का उपयोग
[03] चलिए अपनी प्रस्तुति प्रेजैटेशन sample impress open करते हैं जिसे पिछले tutorial 
[04] चलिए देखते हैं कि screen पर क्या क्या है
[05] मध्य में हम खाली जगह देखते है जोकि workspace है जहाँ हम काम करेंगे
[06] जैसे कि आप देख सकते हैं workspace में 5 tabs हैं जिन्हें view buttons कहते हैं
[07] फिलहाल normal 

In [20]:
EVAL_BATCH_SIZE = 100
test_data   = [dataset["test"][i] for i in range(EVAL_BATCH_SIZE)]
refs_data   = [s["sentence"] for s in test_data]

In [ ]:
# Gemini model
GEMINI_MODEL = "gemini-2.5-flash"

GEMINI_API_KEY = "*****"

# Prompt for transcription
TRANSCRIPTION_PROMPT = """
Generate a verbatim transcript of the speech.
Return only the spoken words.
Do not summarize.
"""
TRANSCRIPTION_PROMPT_HINGLISH = """
Transcribe the speech exactly as spoken.

Important rules:
- Do NOT translate.
- Do NOT convert language.
- If Hindi words are spoken in Hinglish style, write them in Roman script.
- Preserve Hindi-English code switching.
- Return verbatim transcription only.

Example:
Speech: "इस tutorial में हम impress window के भागों"
Output: इस tutorial में हम impress window के भागों
"""

client_gemini = genai.Client(api_key=GEMINI_API_KEY)



In [41]:


wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics_from_lists(predictions, references):
    """Compute WER and CER given prediction and reference lists."""
    # Normalise: strip whitespace, lowercase English parts
    preds_clean = [p.strip() for p in predictions]
    refs_clean  = [r.strip() for r in references]

    wer = wer_metric.compute(predictions=preds_clean, references=refs_clean)
    cer = cer_metric.compute(predictions=preds_clean, references=refs_clean)
    return round(wer * 100, 2), round(cer * 100, 2)

def transcribe_samples(test_samples):
    """
    Transcribe audio samples.
    - If use_pipeline=True  → uses HuggingFace pipeline object
    - If use_pipeline=False → uses model + processor directly (for fine-tuned model)
    """

    client = client_gemini

    predictions = []
    references = []
    sampling_rate = 16000
    count = 0
    for sample in test_samples:
        count += 1
        if((count %2) == 0):
          print(f"\n Processing count {count}")
        audio_array = sample["audio"]["array"].astype(np.float32)

        # Save temp wav file (Sarvam expects file input)
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
            sf.write(tmp_file.name, audio_array, sampling_rate)

            audio_file = client.files.upload(file=tmp_file.name)

            response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[TRANSCRIPTION_PROMPT_HINGLISH, audio_file]
            )
            predicted_text = response.text

        predictions.append(predicted_text)
        reference_text = sample['sentence']
        references.append(reference_text)

    return predictions, references


def print_results_table(model_name, predictions, references):
    """Print a side-by-side comparison table."""
    wer, cer = compute_metrics_from_lists(predictions, references)
    print(f"\n{'='*80}")
    print(f"  Model : {model_name}")
    print(f"  WER   : {wer:.2f}%   |   CER   : {cer:.2f}%")
    print(f"{'='*80}")
    print(f"{'#':<4} {'REFERENCE':<45} {'PREDICTION':<45}")
    print("-" * 96)
    for i, (ref, pred) in enumerate(zip(references, predictions)):
        r = ref[:43] + '…' if len(ref) > 44 else ref
        p = pred[:43] + '…' if len(pred) > 44 else pred
        print(f"{i:<4} {r:<45} {p:<45}")
    print("-" * 96)
    return wer, cer

print("✅ Helper functions defined")

✅ Helper functions defined


In [42]:
print("Running inference on 10 samples...")
preds, ref = transcribe_samples(test_samples)

wer_medium, cer_medium = print_results_table(
    "gemini/gemini-2.5-flash", preds, ref
)

Running inference on 10 samples...

 Processing count 2

 Processing count 4

 Processing count 6

 Processing count 8

 Processing count 10

  Model : gemini/gemini-2.5-flash
  WER   : 47.97%   |   CER   : 40.48%
#    REFERENCE                                     PREDICTION                                   
------------------------------------------------------------------------------------------------
0    लिबर ऑफिस impress में एक प्रस्तुति document…  LibreOffice Impress में एक प्रस्तुति डॉक्यू… 
1    इस tutorial में हम impress window के भागों …  है। इस tutorial में हम impress window के भा… 
2    यहाँ हम अपने ऑपरेटिंग सिस्टम के रूप में gnu…  यहां हम अपने ऑपरेटिंग सिस्टम के रूप में जीए… 
3    चलिए अपनी प्रस्तुति प्रेजैटेशन sample impre…  चलिए अपनी प्रस्तुति सैंपल इंप्रेस ओपन करते … 
4    चलिए देखते हैं कि screen पर क्या क्या है      में बनाया था।
चलिए देखते हैं कि स्क्रीन पर … 
5    मध्य में हम खाली जगह देखते है जोकि workspac…  मध्य में हम खाली जगह देखते हैं जो कि वर्कस्… 
6    जैसे 

In [43]:
print("Running inference on 100 samples...")
preds, ref = transcribe_samples(test_data)

wer_medium_100, cer_medium_100 = print_results_table(
    "gemini/gemini-2.5-flash", preds, ref
)

Running inference on 100 samples...

 Processing count 2

 Processing count 4

 Processing count 6

 Processing count 8

 Processing count 10

 Processing count 12

 Processing count 14

 Processing count 16

 Processing count 18

 Processing count 20

 Processing count 22

 Processing count 24

 Processing count 26

 Processing count 28

 Processing count 30

 Processing count 32

 Processing count 34

 Processing count 36

 Processing count 38

 Processing count 40

 Processing count 42

 Processing count 44

 Processing count 46

 Processing count 48

 Processing count 50

 Processing count 52

 Processing count 54

 Processing count 56

 Processing count 58

 Processing count 60

 Processing count 62

 Processing count 64

 Processing count 66

 Processing count 68

 Processing count 70

 Processing count 72

 Processing count 74

 Processing count 76

 Processing count 78

 Processing count 80

 Processing count 82

 Processing count 84

 Processing count 86

 Processing count 88
